# Selección de regiones para perforación de pozos
Daniel Rodríguez · Ciencia de datos y sector energético

Caso académico de TripleTen. Copia de presentación independiente del original.

**Estado:** código revisado; faltan los CSV para entrenar, ejecutar gráficos y validar resultados. No se publica una región ganadora ni se reutilizan los beneficios y riesgos del original.

## 1. Pregunta de negocio
Comparar tres regiones seleccionando 200 pozos por volumen predicho y evaluando el beneficio simplificado con sus volúmenes reales. Las variables f0, f1 y f2 son anónimas; no se les atribuye significado geológico. No es una implementación empresarial ni un proyecto del Ministerio de Minas y Energía.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from analisis_pozos import load_region, fit_region, profit, bootstrap_profit, summarize, FEATURES, BUDGET, UNIT_REVENUE, N_WELLS
DATA_DIR = Path('data')
# Confirmar estos dos parámetros en el enunciado antes de ejecutar el análisis de riesgo.
CANDIDATES_PER_REPLICA = None
MAX_LOSS_PROBABILITY = None  # Fracción entre 0 y 1.
SEED = 42
N_BOOTSTRAP = 1000

## 2. Calidad y exploración
El cargador valida columnas, ausentes, valores finitos y volúmenes negativos. Si hay identificadores repetidos, detiene el flujo para revisar su significado y definir un tratamiento justificado antes de dividir. No elimina registros automáticamente. Los gráficos se generan a partir de los CSV, sin incorporar cifras ilustrativas.

In [ ]:
regions = {f'Región {i}': load_region(DATA_DIR / f'geo_data_{i}.csv') for i in range(3)}
quality = pd.DataFrame({name: {'filas':len(df), 'IDs repetidos':int(df.id.duplicated().sum()), 'duplicados completos':int(df.duplicated().sum())} for name, df in regions.items()}).T
display(quality)
fig, axes = plt.subplots(1, 3, figsize=(14,4), constrained_layout=True)
for ax, (name, df) in zip(axes, regions.items()):
    ax.hist(df['product'], bins=40, color='#087f8c')
    ax.set(title=name, xlabel='product (miles de barriles, según el caso)', ylabel='Frecuencia')
    print(name)
    display(df[FEATURES + ['product']].describe())
    display(df[FEATURES + ['product']].corr().round(3))
plt.show()

## 3. Entrenamiento y validación
Regresión lineal, partición aleatoria 75/25 y semilla 42, como en el original. Se reportan RMSE y media predicha; no equivalen a rentabilidad. Una dependencia espacial o mediciones repetidas del mismo pozo exigirían rediseñar la partición. Revisar especialmente la procedencia de los datos y la relación casi determinista de variables con el objetivo en la región 1.

In [ ]:
evaluations = {name: fit_region(df) for name, df in regions.items()}
metrics = pd.DataFrame({name: result[2] for name, result in evaluations.items()}).T
display(metrics)
metrics['rmse'].plot.bar(color='#087f8c', ylabel='RMSE (unidades de product)', title='Error en validación', rot=0)
plt.show()

## 4. Evaluación económica
Beneficio simplificado = volumen real de los seleccionados × USD 4.500 − USD 100 millones. El umbral es 111,11 unidades por pozo seleccionado. El promedio regional completo no determina la rentabilidad de una selección de los mejores pozos.

El diagnóstico siguiente selecciona los mejores 200 de **toda** la validación. Es una política distinta del escenario de candidatos del bootstrap y no deben confundirse.

In [ ]:
print(f'Umbral de equilibrio por pozo seleccionado: {BUDGET / N_WELLS / UNIT_REVENUE:.2f} unidades')
display(pd.Series({name:profit(result[1]) for name,result in evaluations.items()}, name='Beneficio diagnóstico USD: mejores 200 de toda la validación').to_frame())

## 5. Distribución de beneficio y riesgo
Cada réplica remuestrea pares completos (observación real y predicción), elige 200 según la predicción y evalúa el volumen real descontando el presupuesto. Se confirma el número de candidatos y el límite de riesgo en el enunciado; no se inventan aquí.

Los percentiles 2,5 y 97,5 describen la distribución bootstrap del beneficio del escenario. No son garantía de rentabilidad ni intervalo de confianza de la media sin análisis adicional. El remuestreo con reemplazo puede repetir observaciones: es un experimento estadístico, no una lista física de pozos únicos.

In [ ]:
if CANDIDATES_PER_REPLICA is None or MAX_LOSS_PROBABILITY is None:
    raise ValueError('Confirma en el enunciado candidatos por réplica y límite de riesgo y completa la configuración.')
if not 0 <= MAX_LOSS_PROBABILITY <= 1:
    raise ValueError('El límite de riesgo debe estar entre 0 y 1.')
distributions = {name:bootstrap_profit(result[1], CANDIDATES_PER_REPLICA, n_boot=N_BOOTSTRAP, seed=SEED) for name,result in evaluations.items()}
risk_table = pd.DataFrame({name:summarize(values) for name,values in distributions.items()}).T
display(risk_table)
fig, axes = plt.subplots(1,3,figsize=(14,4),constrained_layout=True)
for ax,(name,values) in zip(axes,distributions.items()):
    ax.hist(values/1e6,bins=35,color='#087f8c')
    ax.axvline(0,color='#d1495b',linestyle='--',label='Equilibrio')
    ax.set(title=name,xlabel='Beneficio simplificado (millones USD)',ylabel='Réplicas')
    ax.legend()
plt.show()

## 6. Conclusión condicionada
Entre las regiones que cumplan el límite confirmado, identificar la de mayor beneficio medio. Si ninguna cumple, no recomendar región. Revisar sensibilidad al número de candidatos, otras semillas, calidad de datos y supuestos económicos antes de comunicar el resultado.

In [ ]:
eligible = risk_table[risk_table.loss_probability <= MAX_LOSS_PROBABILITY]
if eligible.empty:
    print('Ninguna región cumple el límite de riesgo del escenario.')
else:
    preferred = eligible.mean_profit_usd.idxmax()
    print(f'Candidata según el criterio académico configurado: {preferred}.')
    print('Resultado condicionado a la validación técnica y a los supuestos del caso.')

## 7. Trazabilidad y alcance
El original permanece intacto. La copia elimina comentarios del revisor y conclusiones que el cálculo corregido no respalda. La lógica está en `analisis_pozos.py`; los cambios y las instrucciones de ejecución se documentan en `README.md`.

No hay datos ni resultados de una nueva ejecución en este paquete. El bootstrap mantiene el modelo fijo y no captura toda la incertidumbre de entrenamiento, geológica o económica. El beneficio no es VAN, TIR, flujo de caja descontado ni reservas certificadas. Las dependencias aún no están fijadas mediante una ejecución completa.